# API

Run from the repo root, after `python -m similarity.preprocessing`.

In [ ]:
import time
import pandas as pd
from fastapi.testclient import TestClient
import app as api
from similarity.preprocessing import load_products

products = load_products().set_index("uniq_id", drop=False)
client = TestClient(api.app)

## health

In [ ]:
client.get("/health").json()

## one request

In [ ]:
query = products["uniq_id"].iloc[3]
r = client.get("/find_similar_products", params={"product_id": query, "num_similar": 5})
r.status_code, r.json()

In [ ]:
r.url

In [ ]:
products.loc[[query] + r.json(), ["name", "child_category", "price"]]

## errors

In [ ]:
cases = {
    "unknown id": {"product_id": "not-a-product"},
    "no product_id": {},
    "empty product_id": {"product_id": ""},
    "num_similar 0": {"product_id": query, "num_similar": 0},
    "num_similar 101": {"product_id": query, "num_similar": 101},
    "num_similar ten": {"product_id": query, "num_similar": "ten"},
}
rows = []
for name, params in cases.items():
    r = client.get("/find_similar_products", params=params)
    rows.append({"case": name, "status": r.status_code, "detail": str(r.json()["detail"])[:90]})
pd.DataFrame(rows)

In [ ]:
client.get("/find_similar_products", params={"product_id": query, "num_similar": 101}).json()

## num_similar

In [ ]:
three = client.get("/find_similar_products", params={"product_id": query, "num_similar": 3}).json()
ten = client.get("/find_similar_products", params={"product_id": query, "num_similar": 10}).json()
three == ten[:3]

In [ ]:
len(client.get("/find_similar_products", params={"product_id": query}).json())

## cache

In [ ]:
api.top_similar.cache_clear()
start = time.perf_counter()
client.get("/find_similar_products", params={"product_id": query})
first = time.perf_counter() - start
start = time.perf_counter()
client.get("/find_similar_products", params={"product_id": query})
second = time.perf_counter() - start
round(first * 1000, 1), round(second * 1000, 1)

In [ ]:
api.top_similar.cache_info()

In [ ]:
client.get("/find_similar_products", params={"product_id": "not-a-product"})
api.top_similar.cache_info()

## many requests

In [ ]:
sample = products["uniq_id"].sample(200, random_state=0)
api.top_similar.cache_clear()
start = time.perf_counter()
for product_id in sample:
    client.get("/find_similar_products", params={"product_id": product_id})
cold = (time.perf_counter() - start) / len(sample) * 1000
start = time.perf_counter()
for product_id in sample:
    client.get("/find_similar_products", params={"product_id": product_id})
warm = (time.perf_counter() - start) / len(sample) * 1000
round(cold, 1), round(warm, 1)

## openapi

In [ ]:
list(client.get("/openapi.json").json()["paths"])

In [ ]:
parameters = client.get("/openapi.json").json()["paths"]["/find_similar_products"]["get"]["parameters"]
pd.DataFrame([{"name": p["name"], "required": p["required"], **p["schema"]} for p in parameters])